<a href="https://colab.research.google.com/github/Yiiize/MSSP6070/blob/main/WeeklyModules/Week09/PARTICIPATION_ACTIVITY_Week_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
from google.colab import userdata
import os

github_token = userdata.get('Git_Key')
owner = 'Yiiize' # Replace with the GitHub repository owner
repository = 'MSSP6070' # Replace with the GitHub repository name

clone_url = f'https://{github_token}@github.com/{owner}/{repository}.git'

# Clone the repository
import subprocess
subprocess.run(['git', 'clone', clone_url])

# Navigate into the cloned repository directory (optional, but often useful)
os.chdir(repository)
print(f"Changed directory to: {os.getcwd()}")


Changed directory to: /content/MSSP6070


In [6]:
import os
import requests
from pathlib import Path
from google.colab import userdata
# ==== CONFIG ====
FLICKR_API_KEY = userdata.get('Git_Key')
  # <- put your key here
FLICKR_API_URL = "https://www.flickr.com/services/rest/"
MAX_PHOTOS = 100  # change if you want more/less


def search_flickr_photos(query, per_page=100):
    """
    Generator that yields photo metadata from Flickr search.
    """
    page = 1
    downloaded = 0

    while True:
        params = {
            "method": "flickr.photos.search",
            "api_key": FLICKR_API_KEY,
            "text": query,
            "per_page": per_page,
            "page": page,
            "format": "json",
            "nojsoncallback": 1,
            "content_type": 1,  # photos only
            "safe_search": 1    # safe content
        }

        resp = requests.get(FLICKR_API_URL, params=params)
        resp.raise_for_status()
        data = resp.json()

        photos = data.get("photos", {}).get("photo", [])
        if not photos:
            break

        for p in photos:
            yield p
            downloaded += 1
            if downloaded >= MAX_PHOTOS:
                return

        # move to next page
        page += 1
        if page > data["photos"]["pages"]:
            break


def build_photo_url(photo, size_suffix="b"):
    """
    Build the URL to a Flickr photo.

    size_suffix examples:
      ""  -> default
      "m" -> small
      "b" -> large
    Docs: https://www.flickr.com/services/api/misc.urls.html
    """
    return f"https://live.staticflickr.com/{photo['server']}/{photo['id']}_{photo['secret']}_{size_suffix}.jpg"


def download_image(url, dest_folder):
    dest_folder = Path(dest_folder)
    dest_folder.mkdir(parents=True, exist_ok=True)

    filename = url.split("/")[-1]
    dest_path = dest_folder / filename

    if dest_path.exists():
        print(f"Already exists, skipping: {dest_path}")
        return

    resp = requests.get(url, stream=True)
    resp.raise_for_status()

    with open(dest_path, "wb") as f:
        for chunk in resp.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)

    print(f"Downloaded: {dest_path}")


def main():
    query = input("Enter a photo category to download (e.g., 'cats'): ").strip()
    if not query:
        print("No query entered. Exiting.")
        return

    out_dir = Path("images") / query

    count = 0
    for photo in search_flickr_photos(query):
        url = build_photo_url(photo, size_suffix="b")
        download_image(url, out_dir)
        count += 1

    print(f"\nFinished. Downloaded {count} images for category '{query}' into '{out_dir}'.")


if __name__ == "__main__":
    main()


Enter a photo category to download (e.g., 'cats'): cats

Finished. Downloaded 0 images for category 'cats' into 'images/cats'.


After running the above cell, you will be prompted to authorize Google Drive access. Follow the instructions to complete the process. Once mounted, your Google Drive files will be accessible at `/content/drive`.